In [1]:
import h5py
import numpy as np
import os
import sys
import yaml

In [2]:

sys.path.append('../')

In [3]:
dataset = h5py.File('/home/fsoto/Documents/LCsSSL/data/FPV2/final/dataset.h5', 'r')
dict_info = yaml.safe_load(open('/home/fsoto/Documents/LCsSSL/data/FPV2/final/dict_info.yaml', 'r'))

In [4]:
features = dataset['extracted_feat_2048']
period_column = dict_info['feat_cols'].index('Multiband_period_12')
periods = features[:, period_column]
periods  = np.array(periods)
periods = np.log10(periods)

/tmp/ipykernel_301077/3984596880.py:5: RuntimeWarning: invalid value encountered in log10
  periods = np.log10(periods)


In [5]:
def get_data(periods,fold):
    train = dataset['periodic_train_' + str(fold)]
    test = dataset['periodic_test']
    train = np.array(train)
    test = np.array(test)
    train_periods = periods[train]
    test_periods = periods[test]
    train_labels = dataset['periodic_labels'][train]
    test_labels = dataset['periodic_labels'][test]
    train_labels = np.array(train_labels)
    test_labels = np.array(test_labels)
    return train_periods, train_labels, test_periods, test_labels



In [6]:
def harmonics(num_heads, p_mode="sub_harmonics"):
    out = []

    if p_mode == "one":
        for i in range(num_heads):
            out.append(1)

        return out

    elif p_mode == "two":
        for i in range(num_heads):
            """first and second armonic"""
            if i < num_heads // 2:
                out.append(1)
            else:
                out.append(2)
        return out

    elif p_mode == "geometric":
        for i in range(num_heads):
            """first and second armonic"""
            if i < 2:
                out.append(i + 1)
            else:
                out.append(4**i)

        out = np.array(out)
        out = out[np.argsort(out)]

        return out

    elif p_mode == "sub_harmonics":
        for i in range(num_heads // 2):
            """first and second armonic"""
            out.append(1 / 2**i)

        for i in range(num_heads // 2):
            """first and second armonic"""
            out.append(2**i)

        out = np.array(out)
        out = out[np.argsort(out)]

        return out

    elif p_mode == "mix_harmonics":
        for i in range(num_heads):
            if i < num_heads // 2:
                out.append(2**i)
            else:
                out.append(0)

        out = np.array(out)
        out = out[np.argsort(out)]

        return out

In [7]:
def periods_harmonics(periods, n_harmonics=32):
    sub_harmonics = harmonics(n_harmonics, p_mode="sub_harmonics")
    sub_harmonics = np.array(sub_harmonics)
    
    # Reshape periods to (252810, 1) and sub_harmonics to (1, 32)
    # This allows broadcasting to (252810, 32)
    periods_reshaped = periods.reshape(-1, 1)
    harmonics_reshaped = sub_harmonics.reshape(1, -1)
    
    # Multiply with broadcasting
    result = periods_reshaped * harmonics_reshaped
    
    return result

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, classification_report

import matplotlib.pyplot as plt

# Store metrics for each fold
fold_metrics = {
    'accuracy': [],
    'f1_macro': [],
    'f1_weighted': []
}

# Loop through folds 0-4
for fold in range(5):
    print(f"Processing fold {fold}")
    
    # Get data for this fold
    train_periods, train_labels, test_periods, test_labels = get_data(periods, fold)
    
    # Create harmonic features for training and test data
    train_harmonics = periods_harmonics(train_periods)
    test_harmonics = periods_harmonics(test_periods)
    
    # Create and train XGBoost model
    model = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=10,
        random_state=42
    )
    
    model.fit(train_harmonics, train_labels)
    
    # Make predictions
    y_pred = model.predict(test_harmonics)
    
    # Calculate metrics
    acc = accuracy_score(test_labels, y_pred)
    f1_mac = f1_score(test_labels, y_pred, average='macro')
    f1_wei = f1_score(test_labels, y_pred, average='weighted')
    
    # Store metrics
    fold_metrics['accuracy'].append(acc)
    fold_metrics['f1_macro'].append(f1_mac)
    fold_metrics['f1_weighted'].append(f1_wei)
    
    print(f"Fold {fold} - Accuracy: {acc:.4f}, F1 Macro: {f1_mac:.4f}, F1 Weighted: {f1_wei:.4f}")

# Plot metrics
plt.figure(figsize=(12, 6))

# Plot accuracy
plt.subplot(1, 2, 1)
plt.plot(range(5), fold_metrics['accuracy'], 'o-', label='Accuracy')
plt.title('Accuracy by Fold')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.xticks(range(5))
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()

# Plot F1 scores
plt.subplot(1, 2, 2)
plt.plot(range(5), fold_metrics['f1_macro'], 'o-', label='F1 Macro')
plt.plot(range(5), fold_metrics['f1_weighted'], 'o-', label='F1 Weighted')
plt.title('F1 Scores by Fold')
plt.xlabel('Fold')
plt.ylabel('F1 Score')
plt.xticks(range(5))
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()

plt.tight_layout()
plt.show()

# Print average metrics
print("\nAverage Metrics:")
print(f"Accuracy: {np.mean(fold_metrics['accuracy']):.4f}")
print(f"F1 Macro: {np.mean(fold_metrics['f1_macro']):.4f}")
print(f"F1 Weighted: {np.mean(fold_metrics['f1_weighted']):.4f}")

Processing fold 0
Fold 0 - Accuracy: 0.6632, F1 Macro: 0.5953, F1 Weighted: 0.6371
Processing fold 1
